In [1]:
import duckdb
import pandas as pd
import plotly.express as px
import numpy as np

db = duckdb.connect(
    "C:/Users/Decss/Desktop/retail-transaction-analyses/Data/retail.db",
    read_only=True
)
db.sql("SET search_path TO mart;")
db.sql("""SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'mart'""")

┌─────────────────┐
│   table_name    │
│     varchar     │
├─────────────────┤
│ customerdim     │
│ productdim      │
│ retail_enriched │
└─────────────────┘

Uppdrag: Churn Risk Analysis

Kontext:
Retention-teamet vill minska kundtapp.

Frågor:
Vilka kunder riskerar att sluta handla?


Analys
Använd:

days_since_last_purchase
avg_days_between_orders
purchase_trend

Dela upp kunder efter datakvalitet/tillgänglighet
kunder med:
1 order
2-3 orders
4+ orders

Skapa churn rule:
days_since_last_purchase > 3 * avg_days_between_orders
Eller regel i flera steg? days_since_last_purchase / avg_days_between_orders
Där:
~1 = normal
2+ = avviker tydligt
3+ = churn risk

purchase_trend?

Koppla till RFM-segment i tidigare analys?

Resultat

Identifiera:
high churn risk customers

Visualisera
Histogram - days_since_last_purchase

Rekommendationer
Retention-kampanjer

Personliga erbjudanden?
Rabatter?

Begränsningar:
Kort dataperiod - Svårt eller ej möjligt att skapa prediktiv modell.

In [ ]:
customer_stat = db.sql("""SELECT *
                       FROM customerdim
                       WHERE
                       (customerid NOT IN (
                        SELECT DISTINCT customerid 
                        FROM staging.top_anomalies 
                        WHERE anomaly_type = 'anomaly_customer'
                        ) OR customerid IS NULL)
                       """).df()
customer_stat[['avg_days_between_orders','days_since_last_purchase','total_orders']].describe()

,avg_days_between_orders,days_since_last_purchase,total_orders
count,2847.000000,4339.0,4372.000000
mean,72.465977,92.041484,4.239707
std,65.469811,100.007757,7.685252
min,0.000000,0.0,0.000000
25%,29.366667,17.0,1.000000
50%,53.333333,50.0,2.000000
75%,91.708333,141.5,5.000000
max,366.000000,373.0,210.000000
